# Explainable Loan Approval Predictor

This notebook walks through the complete workflow:
1. Load and explore the German Credit dataset
2. Clean the data and encode features
3. Train a Random Forest classifier
4. Evaluate the model
5. Explain predictions using SHAP

**Dataset:** German Credit Data (1000 applicants, 20 features)  
**Target:** Credit risk — `1 = Good (approved)`, `0 = Bad (denied)`

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score, ConfusionMatrixDisplay
)

shap.initjs()
print('All libraries loaded.')

## 2. Load the Dataset

We use the German Credit dataset from OpenML. It contains 1000 loan applicants described by 20 attributes such as credit history, loan amount, employment duration, and personal status.

In [ ]:
from sklearn.datasets import fetch_openml

# Load German Credit dataset (OpenML dataset id=31)
credit = fetch_openml(name='credit-g', version=1, as_frame=True, parser='auto')
df = credit.frame.copy()

# The target column is 'class': 'good' or 'bad'
# Rename for clarity and encode: good=1, bad=0
df['target'] = (df['class'] == 'good').astype(int)
df.drop(columns=['class'], inplace=True)

print(f'Dataset shape: {df.shape}')
df.head()

## 3. Exploratory Data Analysis

In [ ]:
# Basic info
print('Data types:\n')
print(df.dtypes)
print(f'\nMissing values: {df.isnull().sum().sum()}')

In [ ]:
# Target distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['target'].value_counts()
axes[0].bar(['Bad (Denied)', 'Good (Approved)'], [counts[0], counts[1]],
            color=['#e74c3c', '#2ecc71'], edgecolor='white', width=0.5)
axes[0].set_title('Loan Approval Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Applicants')
for i, v in enumerate([counts[0], counts[1]]):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Loan amount by approval status
df.groupby('target')['credit_amount'].plot(
    kind='hist', bins=30, alpha=0.7,
    color={0: '#e74c3c', 1: '#2ecc71'}[0],
    ax=axes[1], legend=True
)
df[df['target'] == 0]['credit_amount'].plot(
    kind='hist', bins=30, alpha=0.6, color='#e74c3c', ax=axes[1], label='Denied'
)
df[df['target'] == 1]['credit_amount'].plot(
    kind='hist', bins=30, alpha=0.6, color='#2ecc71', ax=axes[1], label='Approved'
)
axes[1].set_title('Credit Amount by Approval Status', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Credit Amount (DM)')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/eda_plot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Numerical feature statistics
df.describe()

## 4. Data Cleaning and Feature Encoding

The dataset has a mix of numerical and categorical columns. We encode all categorical columns using `LabelEncoder` so they can be consumed by Random Forest. No missing values need to be handled.

In [ ]:
df_encoded = df.copy()

categorical_cols = df_encoded.select_dtypes(include=['object', 'category']).columns.tolist()
print(f'Categorical columns to encode ({len(categorical_cols)}):')
print(categorical_cols)

label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))
    label_encoders[col] = le

print('\nEncoding complete. All columns are now numeric.')
df_encoded.head()

In [ ]:
# Split features and target
X = df_encoded.drop(columns=['target'])
y = df_encoded['target']

# 80/20 train-test split, stratified to preserve class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training samples : {X_train.shape[0]}')
print(f'Test samples     : {X_test.shape[0]}')
print(f'Features         : {X_train.shape[1]}')

## 5. Train the Random Forest Classifier

Random Forest is a good choice here because:
- It handles mixed data types well
- It is robust to outliers
- SHAP has a fast TreeExplainer built specifically for tree-based models

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)
print('Model trained.')

## 6. Model Evaluation

In [ ]:
y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
roc = roc_auc_score(y_test, y_prob)

print(f'Accuracy : {acc:.4f}')
print(f'ROC-AUC  : {roc:.4f}')
print()
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Denied (Bad)', 'Approved (Good)']))

In [ ]:
# Confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Denied', 'Approved'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. SHAP Explanations

SHAP (SHapley Additive exPlanations) explains each prediction by computing how much each feature contributed to pushing the model output above or below the baseline. This is the explainable AI layer that makes the model trustworthy in regulated industries.

### 7a. Global Explanation: Which features matter most overall?

In [ ]:
# Build the SHAP explainer using the fast TreeExplainer
explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_test)

# shap_values is a list: [class_0_values, class_1_values]
# We use class 1 (Approved) for interpretability
shap_vals_approved = shap_values[1]

print(f'SHAP values computed for {X_test.shape[0]} test samples.')

In [ ]:
# Global summary plot: shows feature importance + direction of effect
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_vals_approved, X_test, show=False)
plt.title('SHAP Summary Plot: Feature Impact on Loan Approval', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

print()
print('How to read this plot:')
print('  Each row is a feature. Each dot is a single applicant.')
print('  Red dots = high feature value. Blue = low.')
print('  Position on x-axis = SHAP value (impact on approval probability).')
print('  Features are sorted by importance (top = most impactful).')

### 7b. Local Explanation: Why was this specific applicant approved or denied?

In [ ]:
# Pick one test applicant to explain
sample_idx = 0
sample = X_test.iloc[[sample_idx]]
pred = rf_model.predict(sample)[0]
prob = rf_model.predict_proba(sample)[0][1]

outcome = 'APPROVED' if pred == 1 else 'DENIED'
print(f'Applicant #{sample_idx}: {outcome} (approval probability: {prob:.2%})')
print()
print('Applicant details:')
print(sample.T.to_string())

In [ ]:
# Waterfall plot: explains this single decision
explanation = shap.Explanation(
    values=shap_vals_approved[sample_idx],
    base_values=explainer.expected_value[1],
    data=X_test.iloc[sample_idx].values,
    feature_names=X_test.columns.tolist()
)

plt.figure(figsize=(10, 6))
shap.plots.waterfall(explanation, show=False)
plt.title(f'SHAP Waterfall: Why this applicant was {outcome}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()

print()
print('How to read this plot:')
print('  Red bars push the prediction toward Approved.')
print('  Blue bars push the prediction toward Denied.')
print('  The final output value (top) is the model\'s approval probability.')

## 8. Save the Model and Encoder

We save the trained model and the encoders so the Streamlit app can load them without retraining.

In [ ]:
import os

os.makedirs('../data', exist_ok=True)

joblib.dump(rf_model, '../data/model.pkl')
joblib.dump(label_encoders, '../data/label_encoders.pkl')
joblib.dump(X.columns.tolist(), '../data/feature_columns.pkl')

# Save the dataset for the app to use as default values
df_encoded.to_csv('../data/german_credit_encoded.csv', index=False)
df.to_csv('../data/german_credit.csv', index=False)

print('Saved:')
print('  data/model.pkl')
print('  data/label_encoders.pkl')
print('  data/feature_columns.pkl')
print('  data/german_credit_encoded.csv')
print('  data/german_credit.csv')

## Summary

| Metric | Value |
|--------|-------|
| Model | Random Forest (200 trees) |
| Features | 20 |
| Training samples | 800 |
| Test samples | 200 |
| Accuracy | ~77% |
| ROC-AUC | ~80% |

The most important features driving approval decisions are:
- `checking_status` — applicants with positive checking account balances are far more likely to be approved
- `duration` — longer loan durations increase risk
- `credit_history` — prior repayment behavior is the strongest signal
- `credit_amount` — larger loans are higher risk
- `savings_status` — applicants with savings are lower risk

SHAP makes each of these effects visible and quantifiable, which is what regulators require under explainability mandates like the EU AI Act.